# 🚔 Sri Lanka Police AI — Production Notebook (v3.0)

### What this does:
1. **Cell 1** — Install dependencies & restart kernel if needed
2. **Cell 2** — Start Ollama, download model, build `police-ai-master`
3. **Cell 3** — Fine-tune with Unsloth *(skipped if no dataset file attached)*
4. **Cell 4** — Load Surya GPU OCR (multi-API version support)
5. **Cell 5** — Start production Flask API
6. **Cell 6** — Open public Ngrok tunnel + keep-alive loop

### ⚙️ Settings
- **Accelerator**: GPU T4 x2 (recommended) or P100
- **Internet**: Must be ON
- **Dataset** *(optional)*: Attach `.jsonl` to `/kaggle/input` for fine-tuning

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1 — Install Dependencies & Auto-Restart
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, sys

print('🛠️  Checking environment...')

def _install_all():
    os.system('pip uninstall -y numpy transformers unsloth 2>/dev/null')
    os.system(
        'pip install --quiet --no-cache-dir '
        '"numpy<2.0.0" '
        '"transformers>=4.43.0" '
        '"pyngrok==7.2.2" '
        'unsloth '
        'surya-ocr '
        'Pillow '
        'flask '
        'requests'
    )
    os.system('sudo apt-get install -y zstd > /dev/null 2>&1')
    os.system('curl -fsSL https://ollama.com/install.sh | sh')
    print('✅ Install done — restarting kernel...')
    os._exit(0)

try:
    import numpy as np
    if np.__version__.startswith('2.'):
        print(f'⚠️  NumPy {np.__version__} detected — reinstalling...')
        _install_all()
    else:
        # Quick check that surya + flask exist too
        import surya, flask, pyngrok, unsloth
        print(f'✅ NumPy {np.__version__} | All packages OK — no restart needed')
except ImportError as e:
    print(f'⚠️  Missing package ({e}) — installing everything...')
    _install_all()

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — Start Ollama + Build police-ai-master
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import subprocess, time, requests as _req

# --- Start Ollama ---
subprocess.Popen(['ollama', 'serve'],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)
print('✅ Ollama started')

# --- Download base model (skip if cached) ---
try:
    tags = _req.get('http://localhost:11434/api/tags', timeout=5).json()
    existing = [m['name'] for m in tags.get('models', [])]
except Exception:
    existing = []

if 'gemma2:9b' not in existing:
    print('📥 Downloading gemma2:9b (~5 GB first time)...')
    os.system('ollama pull gemma2:9b')
else:
    print('✅ gemma2:9b already cached — skipping download')

# ─── Build Modelfile ──────────────────────────────────────────────────────
# FIX v3: Written line-by-line with string concatenation.
#          This avoids f-string / triple-quote / leading-zero integer
#          SyntaxError bugs that affected v2.x notebooks.

SYSTEM_PROMPT = (
    'You are the Sovereign Sri Lanka Police AI Architect.\n'
    'Your primary directive is Institutional Integrity and Factual Zero-Hallucination.\n'
    'You ONLY output valid JSON. Never add explanation, markdown, or extra text.\n'
    '\n'
    'OFFENCE CLASSIFICATION:\n'
    '- Category 01: Terrorist Activities\n'
    '- Category 02: Recovery of Arms and Ammunition\n'
    '- Category 03: Protests and Strikes\n'
    '- Category 04: SERIOUS CRIMES COMMITTED (Includes Homicides)\n'
    '- Category 05: Robberies\n'
    '- Category 06: Thefts of Vehicles\n'
    '- Category 07: Thefts of Properties\n'
    '- Category 08: House Breaking and Theft\n'
    '- Category 09: Rape, Sexual Abuse and Child Abuse\n'
    '- Category 10: FATAL ACCIDENTS\n'
    '- Category 11: Unidentified dead bodies and suspicious dead bodies\n'
    '- Category 12: Police Accidents\n'
    '- Category 13: Serious injuries of Police officers and Damages to Police property\n'
    '- Category 14: Misconducts of Police officers\n'
    '- Category 15: Deaths of Police officers\n'
    '- Category 16: Hospital admission of SGOFO\n'
    '- Category 17: Passing away of close relatives of SGOFO\n'
    '- Category 18: Passing away of close relatives of retired SGOFO\n'
    '- Category 19: Detentions of Narcotics and Illicit Liquor\n'
    '- Category 20: Arrests\n'
    '- Category 21: Arresting of Tri-Forces Members\n'
    '- Category 22: Disappearances\n'
    '- Category 23: Suicides\n'
    '- Category 24: Incidents regarding Foreigners\n'
    '- Category 25: Wild elephant attacks and deaths of wild elephants\n'
    '- Category 26: Deaths due to drowning\n'
    '- Category 27: Incidents of Fire\n'
    '- Category 28: Other Matters\n'
    '\n'
    'EXTRACTION PROTOCOL:\n'
    '- DATE: YYYY-MM-DD format. Default: "N/A"\n'
    '- TIME: 24-hour HHMM format (e.g. 1430). Default: "N/A"\n'
    '- DESCRIPTION: Formal, clinical Police English. No assumptions.\n'
    '- REPETITION CONTROL: Never repeat phrases or sentences.\n'
    '- FINANCIAL_LOSS: Numeric string in LKR. Use "N/A" if none.\n'
    '- STATUS: One of: Arrested | Fled | Under Investigation | Recovered | Unknown\n'
    '- CATEGORY: Use numeric code only (e.g. "04")\n'
    '\n'
    'JSON OUTPUT SCHEMA (return ONLY this, nothing else):\n'
    '{\n'
    '  "station": "",\n'
    '  "division": "",\n'
    '  "date": "",\n'
    '  "time": "",\n'
    '  "category": "",\n'
    '  "description": "",\n'
    '  "financial_loss": "",\n'
    '  "status": "",\n'
    '  "victim_names": [],\n'
    '  "suspect_names": [],\n'
    '  "vehicle_numbers": [],\n'
    '  "locations": []\n'
    '}'
)

mf  = 'FROM gemma2:9b\n'
mf += 'PARAMETER temperature 0.05\n'
mf += 'PARAMETER num_ctx 32768\n'
mf += 'PARAMETER top_p 0.9\n'
mf += 'PARAMETER repeat_penalty 1.5\n'
mf += 'PARAMETER stop "<|im_start|>"\n'
mf += 'PARAMETER stop "<|im_end|>"\n'
mf += 'PARAMETER stop "<|file_separator|>"\n'
mf += 'SYSTEM """\n'
mf += SYSTEM_PROMPT
mf += '\n"""\n'

with open('Modelfile', 'w', encoding='utf-8') as f:
    f.write(mf)

print('\n--- Modelfile preview (first 12 lines) ---')
with open('Modelfile') as f:
    for i, line in enumerate(f):
        if i < 12:
            print(f'  {line}', end='')
print('\n---')

print('\n🔨 Building police-ai-master:latest...')
os.system('ollama create police-ai-master:latest -f Modelfile')

result = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
print(result.stdout)
if 'police-ai-master' in result.stdout:
    print('✅ police-ai-master is ready!')
else:
    print('❌ Model creation failed — review Modelfile preview above')

import os

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3 — Fine-Tune with Unsloth (skipped if no JSONL dataset found)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import glob, os

# Search for training JSONL — in Kaggle /input or current directory
candidates = (
    glob.glob('/kaggle/input/**/*.jsonl', recursive=True) +
    glob.glob('*.jsonl')
)
# Filter out generated output files
candidates = [f for f in candidates if 'output' not in f.lower()]

if not candidates:
    print('⚠️  No JSONL training data found — skipping fine-tuning.')
    print('   To fine-tune: attach a .jsonl dataset file to /kaggle/input')
else:
    TRAIN_PATH = candidates[0]
    print(f'📂 Training dataset: {TRAIN_PATH}')

    from unsloth import FastLanguageModel
    from trl import SFTTrainer
    from transformers import TrainingArguments
    from datasets import load_dataset
    import torch

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = 'unsloth/gemma-2-9b-it-bnb-4bit',
        max_seq_length = 2048,
        load_in_4bit   = True,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r              = 32,
        lora_alpha     = 16,
        lora_dropout   = 0,
        target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                          'gate_proj', 'up_proj', 'down_proj'],
        use_gradient_checkpointing = 'unsloth',
        random_state   = 3407,
    )

    dataset = load_dataset('json', data_files=TRAIN_PATH, split='train')
    dataset = dataset.map(
        lambda e: {'text': [tokenizer.apply_chat_template(c, tokenize=False)
                            for c in e['messages']]},
        batched=True
    )

    trainer = SFTTrainer(
        model              = model,
        tokenizer          = tokenizer,
        train_dataset      = dataset,
        dataset_text_field = 'text',
        max_seq_length     = 2048,
        args = TrainingArguments(
            per_device_train_batch_size  = 2,
            gradient_accumulation_steps  = 4,
            warmup_steps                 = 5,
            max_steps                    = 60,
            learning_rate                = 2e-4,
            fp16  = not torch.cuda.is_bf16_supported(),
            bf16  = torch.cuda.is_bf16_supported(),
            logging_steps  = 1,
            optim          = 'adamw_8bit',
            output_dir     = 'output',
        )
    )

    print('🧠 Training started...')
    trainer.train()

    GGUF_DIR = 'police-ai-master-gguf'
    print(f'📦 Exporting to GGUF: {GGUF_DIR}')
    model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method='q4_k_m')
    print('✅ Fine-tuning complete!')

    # Rebuild model from fine-tuned GGUF
    gguf_files = glob.glob(f'{GGUF_DIR}/*.gguf')
    if gguf_files:
        gguf_path = gguf_files[0]
        mf2  = f'FROM {gguf_path}\n'
        mf2 += 'PARAMETER temperature 0.05\n'
        mf2 += 'PARAMETER repeat_penalty 1.5\n'
        mf2 += 'PARAMETER num_ctx 32768\n'
        mf2 += 'SYSTEM """\n'
        mf2 += SYSTEM_PROMPT
        mf2 += '\n"""\n'
        with open('Modelfile', 'w', encoding='utf-8') as f:
            f.write(mf2)
        os.system('ollama create police-ai-master:latest -f Modelfile')
        print('✅ Rebuilt police-ai-master from fine-tuned GGUF!')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 4 — Load Surya GPU OCR
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch, time as _time

print('🧠 Loading Surya OCR into GPU...')
t0 = _time.time()

SURYA_LOADED          = False
recognition_predictor = None
detection_predictor   = None
foundation_predictor  = None
surya_api_version     = None

# --- Try NEW API (surya >= 0.8) ---
try:
    from surya.recognition import RecognitionPredictor
    from surya.detection   import DetectionPredictor

    try:
        from surya.foundation import FoundationPredictor
        foundation_predictor  = FoundationPredictor()
        recognition_predictor = RecognitionPredictor(foundation_predictor)
        surya_api_version = 'new-foundation'
    except ImportError:
        recognition_predictor = RecognitionPredictor()
        surya_api_version = 'new-simple'

    detection_predictor = DetectionPredictor()
    SURYA_LOADED = True
    print(f'✅ Surya API: {surya_api_version}')

except Exception as e1:
    print(f'⚠️  New Surya API failed ({e1}) — trying legacy API...')

    try:
        from surya.model.detection.model       import load_model as load_det_model, load_processor as load_det_processor
        from surya.model.recognition.model     import load_model as load_rec_model
        from surya.model.recognition.processor import load_processor as load_rec_processor
        from surya.ocr import run_ocr

        det_processor = load_det_processor()
        det_model     = load_det_model()
        rec_model     = load_rec_model()
        rec_processor = load_rec_processor()

        class _LegacyOCR:
            def __call__(self, images, langs=None, **kwargs):
                return run_ocr(images, [langs or ['en', 'si']] * len(images),
                               det_model, det_processor, rec_model, rec_processor)

        recognition_predictor = _LegacyOCR()
        detection_predictor   = None
        SURYA_LOADED          = True
        surya_api_version     = 'legacy'
        print('✅ Surya API: legacy')

    except Exception as e2:
        print(f'❌ Both Surya APIs failed.\n  New: {e1}\n  Old: {e2}')
        SURYA_LOADED = False

elapsed = _time.time() - t0
if SURYA_LOADED:
    print(f'✅ Surya OCR loaded in {elapsed:.1f}s')
else:
    print('⚠️  Running without OCR — AI endpoint still works')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 5 — Production Flask API
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import threading, io, json, traceback, logging, datetime
from collections import defaultdict
from flask import Flask, request, jsonify, Response
from PIL   import Image
import requests as _req

logging.basicConfig(
    level   = logging.INFO,
    format  = '%(asctime)s  %(levelname)-7s  %(message)s',
    datefmt = '%H:%M:%S'
)
log = logging.getLogger('police-ai')

app      = Flask(__name__)
gpu_lock = threading.Lock()

MAX_IMAGE_BYTES = 20 * 1024 * 1024
OLLAMA_BASE     = 'http://localhost:11434'

_stats      = defaultdict(int)
_start_time = datetime.datetime.utcnow()

def _bump(key): _stats[key] += 1

# ── Helpers ───────────────────────────────────────────────────────────────
def _extract_surya_text(predictions):
    lines = []
    if not predictions:
        return ''
    page = predictions[0]
    if hasattr(page, 'text_lines'):
        for line in page.text_lines:
            txt = getattr(line, 'text', None) or (line.get('text', '') if isinstance(line, dict) else '')
            if txt and txt.strip(): lines.append(txt.strip())
    elif isinstance(page, dict):
        for line in page.get('text_lines', []):
            txt = line.get('text', '') if isinstance(line, dict) else getattr(line, 'text', '')
            if txt and txt.strip(): lines.append(txt.strip())
    elif isinstance(page, list):
        for item in page:
            txt = item if isinstance(item, str) else getattr(item, 'text', '')
            if txt and txt.strip(): lines.append(txt.strip())
    return '\n'.join(lines)

def _ollama_post(endpoint, data, stream=False, timeout=300):
    import time
    url = f"{OLLAMA_BASE}/{endpoint.lstrip('/')}"
    for attempt in range(3):
        try:
            return _req.post(url, json=data, timeout=timeout, stream=stream)
        except _req.exceptions.ConnectionError:
            if attempt < 2:
                log.warning(f'Ollama connection failed (attempt {attempt+1}), retrying...')
                time.sleep(2)
            else:
                raise

# ━━ Routes ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
@app.route('/', methods=['GET'])
def home():
    try:
        models = [m['name'] for m in _req.get(f'{OLLAMA_BASE}/api/tags', timeout=3).json().get('models', [])]
    except Exception:
        models = []
    return jsonify({
        'service'   : 'Sri Lanka Police AI Cloud v3.0',
        'status'    : 'online',
        'surya_ocr' : SURYA_LOADED,
        'surya_api' : surya_api_version,
        'models'    : models,
        'endpoints' : ['/health', '/stats', '/gpu-ocr', '/process',
                       '/api/generate', '/api/chat', '/api/tags']
    })

@app.route('/health', methods=['GET'])
def health():
    try:
        _req.get(f'{OLLAMA_BASE}/api/tags', timeout=3)
        ollama_ok = True
    except Exception:
        ollama_ok = False
    return jsonify({
        'status'         : 'ok' if ollama_ok else 'degraded',
        'surya'          : SURYA_LOADED,
        'ollama'         : ollama_ok,
        'gpu_mem_used_gb': round(torch.cuda.memory_allocated(0) / 1e9, 2) if torch.cuda.is_available() else None
    })

@app.route('/stats', methods=['GET'])
def stats():
    uptime = (datetime.datetime.utcnow() - _start_time).total_seconds()
    return jsonify({'uptime_minutes': round(uptime / 60, 1), 'requests': dict(_stats)})

@app.route('/gpu-ocr', methods=['POST'])
def gpu_ocr():
    _bump('ocr')
    try:
        if not SURYA_LOADED:
            return jsonify({'success': False, 'error': 'Surya OCR not loaded'}), 503
        if 'file' not in request.files:
            return jsonify({'success': False, 'error': 'Send image as file in multipart form'}), 400
        raw = request.files['file'].read()
        if len(raw) < 100:
            return jsonify({'success': False, 'error': 'File too small'}), 400
        if len(raw) > MAX_IMAGE_BYTES:
            return jsonify({'success': False, 'error': 'File too large (max 20 MB)'}), 413
        image = Image.open(io.BytesIO(raw)).convert('RGB')
        t0 = _time.time()
        with gpu_lock:
            if surya_api_version == 'legacy':
                predictions = recognition_predictor([image])
            elif detection_predictor is not None:
                predictions = recognition_predictor([image], det_predictor=detection_predictor)
            else:
                predictions = recognition_predictor([image])
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        text    = _extract_surya_text(predictions)
        elapsed = round(_time.time() - t0, 2)
        lines   = len([l for l in text.split('\n') if l.strip()])
        log.info(f'[OCR] {lines} lines | {len(text)} chars | {elapsed}s')
        return jsonify({'success': True, 'text': text, 'lines': lines, 'time_sec': elapsed})
    except Exception as e:
        traceback.print_exc()
        _bump('ocr_error')
        return jsonify({'success': False, 'error': str(e)}), 500

@app.route('/process', methods=['POST'])
def process():
    _bump('process')
    try:
        if not SURYA_LOADED:
            return jsonify({'success': False, 'error': 'Surya OCR not loaded'}), 503
        if 'file' not in request.files:
            return jsonify({'success': False, 'error': 'Send image as file in multipart form'}), 400
        raw = request.files['file'].read()
        if len(raw) < 100: return jsonify({'success': False, 'error': 'File too small'}), 400
        if len(raw) > MAX_IMAGE_BYTES: return jsonify({'success': False, 'error': 'File too large'}), 413
        image = Image.open(io.BytesIO(raw)).convert('RGB')
        t0 = _time.time()
        with gpu_lock:
            if surya_api_version == 'legacy':
                predictions = recognition_predictor([image])
            elif detection_predictor is not None:
                predictions = recognition_predictor([image], det_predictor=detection_predictor)
            else:
                predictions = recognition_predictor([image])
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        ocr_text = _extract_surya_text(predictions)
        ocr_time = round(_time.time() - t0, 2)
        model  = request.form.get('model', 'police-ai-master:latest')
        prompt = request.form.get('prompt') or (
            f'Extract all police report fields from the following OCR text and return ONLY valid JSON:\n\n{ocr_text}'
        )
        t1   = _time.time()
        resp = _ollama_post('/api/generate', {'model': model, 'prompt': prompt, 'stream': False}, timeout=120)
        ai_time = round(_time.time() - t1, 2)
        raw_ai  = resp.json().get('response', '')
        try:
            clean   = raw_ai.strip().lstrip('```json').lstrip('```').rstrip('```').strip()
            ai_json = json.loads(clean)
        except Exception:
            ai_json = {'raw_response': raw_ai}
        return jsonify({'success': True, 'ocr_text': ocr_text, 'ai_json': ai_json,
                        'ocr_time': ocr_time, 'ai_time': ai_time})
    except Exception as e:
        traceback.print_exc()
        _bump('process_error')
        return jsonify({'success': False, 'error': str(e)}), 500

@app.route('/api/generate', methods=['POST'])
def ollama_generate():
    _bump('generate')
    try:
        data = request.get_json()
        if not data: return jsonify({'error': 'JSON body required'}), 400
        is_stream = data.get('stream', False)
        resp = _ollama_post('/api/generate', data, stream=is_stream, timeout=300)
        if is_stream:
            return Response((c for c in resp.iter_content(1024) if c),
                            content_type=resp.headers.get('Content-Type', 'application/json'))
        return jsonify(resp.json()), resp.status_code
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/api/chat', methods=['POST'])
def ollama_chat():
    _bump('chat')
    try:
        data = request.get_json()
        if not data: return jsonify({'error': 'JSON body required'}), 400
        is_stream = data.get('stream', False)
        resp = _ollama_post('/api/chat', data, stream=is_stream, timeout=300)
        if is_stream:
            return Response((c for c in resp.iter_content(1024) if c),
                            content_type=resp.headers.get('Content-Type', 'application/json'))
        return jsonify(resp.json()), resp.status_code
    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/api/tags', methods=['GET'])
def ollama_tags():
    try:
        resp = _req.get(f'{OLLAMA_BASE}/api/tags', timeout=10)
        return jsonify(resp.json()), resp.status_code
    except Exception as e:
        return jsonify({'error': str(e)}), 500

print('✅ Flask routes registered:')
print('   GET  /              → Info')
print('   GET  /health        → Health check')
print('   GET  /stats         → Request stats')
print('   POST /gpu-ocr       → Surya OCR only')
print('   POST /process       → OCR + AI extract  ← USE THIS')
print('   POST /api/generate  → Ollama generate')
print('   POST /api/chat      → Ollama chat')
print('   GET  /api/tags      → List models')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 6 — Ngrok Tunnel + Keep-Alive
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ⚠️  YOUR NGROK TOKEN: paste below
NGROK_AUTH_TOKEN = '3C0C7gFkX4IQuTy2cMMPHYznbNh_4CZ5YG6ekExX6sBKNfhpv'

from pyngrok import ngrok
import time, threading

PORT = 5050

flask_thread = threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=PORT, debug=False, use_reloader=False),
    daemon=True
)
flask_thread.start()
time.sleep(3)

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

tunnel = ngrok.connect(PORT)
URL    = tunnel.public_url

print()
print('=' * 70)
print(' 🚔  POLICE AI CLOUD SERVER — LIVE!  🚔 '.center(70))
print('=' * 70)
print(f'  🌐  Public URL : {URL}')
print()
print(f'  📡  OCR only   : POST  {URL}/gpu-ocr')
print(f'  🔁  OCR + AI   : POST  {URL}/process          ← USE THIS')
print(f'  🤖  AI only    : POST  {URL}/api/generate')
print(f'  💬  Chat       : POST  {URL}/api/chat')
print(f'  ❤️   Health     : GET   {URL}/health')
print(f'  📊  Stats      : GET   {URL}/stats')
print()
print(f"  Surya OCR : {'✅ Ready  (' + str(surya_api_version) + ')' if SURYA_LOADED else '❌ Failed'}")
print(f'  Ollama    : ✅ police-ai-master (gemma2:9b)')
print()
print('  ── config.json ──────────────────────────────────────────────────')
print(f'  "kaggle_url"     : "{URL}/api/generate"')
print(f'  "kaggle_ocr_url" : "{URL}/gpu-ocr"')
print(f'  "kaggle_process_url" : "{URL}/process"')
print('  ─────────────────────────────────────────────────────────────────')
print()
print('  ⚠️  Do not close this tab — server will stop!')
print('=' * 70)

n = 0
while True:
    time.sleep(30)
    n += 1
    mins = n * 30 // 60
    if n % 10 == 0:  # every 5 min
        try:
            h = _req.get(f'http://localhost:{PORT}/health', timeout=5).json()
            gpu_info = ''
            if torch.cuda.is_available():
                used     = torch.cuda.memory_allocated(0) / 1e9
                gpu_info = f'| GPU {used:.1f}GB used'
            print(f'  💓 {mins}min running {gpu_info} | {URL}')
        except Exception:
            print(f'  💓 {mins}min running | {URL}')